# 01 Dataset Inventory

This notebook inventories the raw datasets available in the project workspace.

It will:
- scan `data/raw/datasets/`
- list files and subfolders
- preview tabular dataset files
- record basic metadata like rows, columns, and file sizes
- export summary CSVs to `outputs/`


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)


In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATASETS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'datasets'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASETS_DIR =', DATASETS_DIR)
print('OUTPUT_DIR    =', OUTPUT_DIR)

assert DATASETS_DIR.exists(), f'Datasets directory not found: {DATASETS_DIR}'


PROJECT_ROOT = /kaggle/working/CyberBully_Detection_Paper
DATASETS_DIR = /kaggle/working/CyberBully_Detection_Paper/data/raw/datasets
OUTPUT_DIR    = /kaggle/working/CyberBully_Detection_Paper/outputs


## 1. Show dataset folders


In [3]:
dataset_dirs = sorted([p for p in DATASETS_DIR.iterdir() if p.is_dir()])
dataset_names = [p.name for p in dataset_dirs]

display(pd.DataFrame({'dataset_folder': dataset_names}))


,dataset_folder
0,banth
1,bd_shs
2,facebook_44001
3,multilabel_12557


## 2. File tree summary


In [4]:
records = []

for dataset_dir in dataset_dirs:
    for path in sorted(dataset_dir.rglob('*')):
        if path.is_file():
            records.append({
                'dataset_folder': dataset_dir.name,
                'relative_path': str(path.relative_to(DATASETS_DIR)),
                'filename': path.name,
                'suffix': path.suffix.lower(),
                'size_kb': round(path.stat().st_size / 1024, 2),
            })

files_df = pd.DataFrame(records)
files_df = files_df.sort_values(['dataset_folder', 'relative_path']).reset_index(drop=True)
display(files_df)


,dataset_folder,relative_path,filename,suffix,size_kb
0,banth,banth/full_with_stats.csv,full_with_stats.csv,.csv,47193.98
1,banth,banth/test.csv,test.csv,.csv,995.44
2,banth,banth/train.csv,train.csv,.csv,8035.49
3,banth,banth/val.csv,val.csv,.csv,1020.47
4,bd_shs,bd_shs/test.csv,test.csv,.csv,977.13
5,facebook_44001,facebook_44001/bangla_online_comments_dataset.xlsx,bangla_online_comments_dataset.xlsx,.xlsx,4591.26
6,multilabel_12557,multilabel_12557/Multilablel_Cyberbully_Data.csv,Multilablel_Cyberbully_Data.csv,.csv,1638.06


## 3. Find tabular dataset files


In [5]:
TABULAR_EXTS = {'.csv', '.xlsx', '.xls', '.tsv'}

tabular_df = files_df[files_df['suffix'].isin(TABULAR_EXTS)].copy()
tabular_df = tabular_df.reset_index(drop=True)
display(tabular_df)


,dataset_folder,relative_path,filename,suffix,size_kb
0,banth,banth/full_with_stats.csv,full_with_stats.csv,.csv,47193.98
1,banth,banth/test.csv,test.csv,.csv,995.44
2,banth,banth/train.csv,train.csv,.csv,8035.49
3,banth,banth/val.csv,val.csv,.csv,1020.47
4,bd_shs,bd_shs/test.csv,test.csv,.csv,977.13
5,facebook_44001,facebook_44001/bangla_online_comments_dataset.xlsx,bangla_online_comments_dataset.xlsx,.xlsx,4591.26
6,multilabel_12557,multilabel_12557/Multilablel_Cyberbully_Data.csv,Multilablel_Cyberbully_Data.csv,.csv,1638.06


## 4. Read each dataset file and collect metadata


In [6]:
def load_table(path: Path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    elif suffix == '.tsv':
        return pd.read_csv(path, sep='\t')
    elif suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    else:
        raise ValueError(f'Unsupported file type: {suffix}')

inventory_rows = []
preview_tables = {}

for _, row in tabular_df.iterrows():
    file_path = DATASETS_DIR / row['relative_path']
    try:
        df = load_table(file_path)
        preview_tables[row['relative_path']] = df.head(5)
        inventory_rows.append({
            'dataset_folder': row['dataset_folder'],
            'relative_path': row['relative_path'],
            'filename': row['filename'],
            'rows': len(df),
            'cols': len(df.columns),
            'columns': ' | '.join(map(str, df.columns.tolist())),
            'missing_values_total': int(df.isna().sum().sum()),
            'duplicate_rows': int(df.duplicated().sum()),
        })
    except Exception as e:
        inventory_rows.append({
            'dataset_folder': row['dataset_folder'],
            'relative_path': row['relative_path'],
            'filename': row['filename'],
            'rows': None,
            'cols': None,
            'columns': f'ERROR: {e}',
            'missing_values_total': None,
            'duplicate_rows': None,
        })

inventory_df = pd.DataFrame(inventory_rows)
inventory_df = inventory_df.sort_values(['dataset_folder', 'relative_path']).reset_index(drop=True)
display(inventory_df)


,dataset_folder,relative_path,filename,rows,cols,columns,missing_values_total,duplicate_rows
0,banth,banth/full_with_stats.csv,full_with_stats.csv,36649,26,author | updated_at | comment_like_count | Text | video_id | timestamp | Label | Political | Religious | Gender | Personal Offense | Abusive/Violence | Origin | Body Shaming | Misc | title | categ...,1696,0
1,banth,banth/test.csv,test.csv,3735,12,Text | Label | Political | Religious | Gender | Personal Offense | Abusive/Violence | Origin | Body Shaming | Misc | bangla | english,0,0
2,banth,banth/train.csv,train.csv,29879,12,Text | Label | Political | Religious | Gender | Personal Offense | Abusive/Violence | Origin | Body Shaming | Misc | bangla | english,0,0
3,banth,banth/val.csv,val.csv,3736,12,Text | Label | Political | Religious | Gender | Personal Offense | Abusive/Violence | Origin | Body Shaming | Misc | bangla | english,0,0
4,bd_shs,bd_shs/test.csv,test.csv,5029,4,sentence | target | type | hate speech,5226,0
5,facebook_44001,facebook_44001/bangla_online_comments_dataset.xlsx,bangla_online_comments_dataset.xlsx,44001,5,comment | Category | Gender | comment react number | label,3,140
6,multilabel_12557,multilabel_12557/Multilablel_Cyberbully_Data.csv,Multilablel_Cyberbully_Data.csv,12546,8,Gender | Profession | comment | bully | sexual | religious | threat | spam,0,1291


## 5. Preview first rows of each readable dataset


In [7]:
for rel_path, preview_df in preview_tables.items():
    display(Markdown(f'### `{rel_path}`'))
    display(preview_df)


### `banth/full_with_stats.csv`

,author,updated_at,comment_like_count,Text,video_id,timestamp,Label,Political,Religious,Gender,Personal Offense,Abusive/Violence,Origin,Body Shaming,Misc,title,category_id,category_name,channel_id,channel_name,channel_subscribers,channel_description,view_count,video_like_count,comment_count,publish_date
0,@MdEmon-un8tx,2024-03-27T09:25:43Z,1,Moja paici anek😁😁😂😂,ba7Qg81nUvo,2024-03-27 09:25:43+00:00,0,0,0,0,0,0,0,0,0,"ডাক্তার দাবি করা ভাইরাল মুনিয়া জানেন না ওটি, আইসিইউ কি? | Nagorik TV",25,News & Politics,UCig6MgICx3g8p3IHZM45oEA,Nagorik TV,2220000,"Nagorik TV is Bangladesh Government-approved satellite television channel, bringing you a diverse range of content under the motto ""Television Noy, Shomporko"" (Not just TV, but Connection). Found...",3208092,49149.0,15426,2024-03-25T16:20:22Z
1,@sobuzuddin5432,2024-03-27T09:23:29Z,0,Haha ...ki likhe dilen 🤔 sorry?,ba7Qg81nUvo,2024-03-27 09:23:29+00:00,0,0,0,0,0,0,0,0,0,"ডাক্তার দাবি করা ভাইরাল মুনিয়া জানেন না ওটি, আইসিইউ কি? | Nagorik TV",25,News & Politics,UCig6MgICx3g8p3IHZM45oEA,Nagorik TV,2220000,"Nagorik TV is Bangladesh Government-approved satellite television channel, bringing you a diverse range of content under the motto ""Television Noy, Shomporko"" (Not just TV, but Connection). Found...",3208092,49149.0,15426,2024-03-25T16:20:22Z
2,@tonmoy7068,2024-03-27T09:22:53Z,0,Muniyar piles operation kora dorkar😂😂,ba7Qg81nUvo,2024-03-27 09:22:53+00:00,0,0,0,0,0,0,0,0,0,"ডাক্তার দাবি করা ভাইরাল মুনিয়া জানেন না ওটি, আইসিইউ কি? | Nagorik TV",25,News & Politics,UCig6MgICx3g8p3IHZM45oEA,Nagorik TV,2220000,"Nagorik TV is Bangladesh Government-approved satellite television channel, bringing you a diverse range of content under the motto ""Television Noy, Shomporko"" (Not just TV, but Connection). Found...",3208092,49149.0,15426,2024-03-25T16:20:22Z
3,@lailakarin4996,2024-03-27T09:19:46Z,0,Erokom doctor ekmatro Bangladesh pao jabey one piece only,ba7Qg81nUvo,2024-03-27 09:19:46+00:00,1,0,0,0,0,0,0,0,1,"ডাক্তার দাবি করা ভাইরাল মুনিয়া জানেন না ওটি, আইসিইউ কি? | Nagorik TV",25,News & Politics,UCig6MgICx3g8p3IHZM45oEA,Nagorik TV,2220000,"Nagorik TV is Bangladesh Government-approved satellite television channel, bringing you a diverse range of content under the motto ""Television Noy, Shomporko"" (Not just TV, but Connection). Found...",3208092,49149.0,15426,2024-03-25T16:20:22Z
4,@ramkrishna7281,2024-03-27T09:18:29Z,0,Occupational therapy(OT) Intensive care unit (ICU),ba7Qg81nUvo,2024-03-27 09:18:29+00:00,0,0,0,0,0,0,0,0,0,"ডাক্তার দাবি করা ভাইরাল মুনিয়া জানেন না ওটি, আইসিইউ কি? | Nagorik TV",25,News & Politics,UCig6MgICx3g8p3IHZM45oEA,Nagorik TV,2220000,"Nagorik TV is Bangladesh Government-approved satellite television channel, bringing you a diverse range of content under the motto ""Television Noy, Shomporko"" (Not just TV, but Connection). Found...",3208092,49149.0,15426,2024-03-25T16:20:22Z


### `banth/test.csv`

,Text,Label,Political,Religious,Gender,Personal Offense,Abusive/Violence,Origin,Body Shaming,Misc,bangla,english
0,Police 20 takar o ghush Khaile chakri theke Suspend ai niyom Koren Dekhi paren kina ...,1,0,0,0,1,0,0,0,0,পুলিশ ২০ টাকা ও ঘুস খাইলে চাকরি থেকে সাসপেন্ড এই নিয়ম করেন দেখি পারেন কিনা...,"If a policeman gets 20 rupees and gets bribes, make this rule and see if you can..."
1,"Apnara ki gaja kheye news koren ? Sotti Mitha jachai na kore e news kore den 😡 era chatro lig na ex student , student ra kichu teacher der baad deyar kotha bolcelo , oi teacher der pokhe ai ex stu...",1,0,0,0,1,0,0,0,0,"আপনারা কি গাজা খেয়ে নিউজ করেন? সত্যি মিঠা যাই না করে এ নিউজ করে দেন এরা ছাত্র লীগ না এক্স স্টুডেন্ট, স্টুডেন্ট রা কিছু টিচার দের বাদ দেয়ার কথা বললো, ওই টিচার দের চোখে এই এক্স স্টুডেন্ট রা আসছে ...","Do you news by drinking Gaza? Instead of telling me that this is not the student League or X students, the students asked for the exclusion of some teachers, in the eyes of those teachers these X..."
2,Kothay gela khomota go,0,0,0,0,0,0,0,0,0,কোথায় গেলে সুবিধা গো,Where's the advantage?
3,Myb eta India cal parar meye,1,0,0,0,1,0,1,0,0,মেইবি এটা ইন্ডিয়া চাল পাড়ার মেয়ে,This is India rice girl.
4,Tomar punishment sure 😂😂,0,0,0,0,0,0,0,0,0,তোমার শাস্তি নিশ্চিত😂😂,Your punishment is sure.😂😂


### `banth/train.csv`

...

,sentence,target,type,hate speech
0,। গরুর বাচ্চা কুকুরের বাচ্চা,ind,slander,1
1,অন্যায় দাবি করছে ক্রিকেটাররা অথচ দোষ নাকি বোর্ডের ? বাংলাদেশ মাতালের কারখানায় পরিনত হয়েছে।,ind,religion,1
2,আইসিসিকে এই তথ্য দিলো কে?শুওরের বাচ্চা পাপন,male,slander,1
3,আওয়ামীলীগ ভারতের দালাল। মোদি নির্বাচিত হওয়ার পর একজন লীগ সমর্থককেও দেখলাম না মোদিকে অভিনন্দন জানাতে অথচ সবার আগে মোদিকে শুভেচ্ছা জানিয়েছিলো জামায়াত ইসলাম আর নিজেদের রাজনৈতিক কার্যালয়ে চেয়ারপারসনের...,group,slander,1
4,আগারওয়াল পিউর বাস্টার্ড এর কিছু হবে না???,ind,slander,1


### `facebook_44001/bangla_online_comments_dataset.xlsx`

,comment,Category,Gender,comment react number,label
0,ওই হালার পুত এখন কি মদ খাওয়ার সময় রাতের বেলা মদ খাই দিনের বেলাও মাঝেমধ্যে খায় এখন ম*** চ**** সময় safa কে একটু চুদাম যার ইচ্ছা আছে চুদার লাইনে দারা একজন একজন করে জাবি,Actor,Female,1.0,sexual
1,ঘরে বসে শুট করতে কেমন লেগেছে? ক্যামেরাতে কে ছিলেন?,Singer,Male,2.0,not bully
2,"অরে বাবা, এই টা কোন পাগল????",Actor,Female,2.0,not bully
3,ক্যাপ্টেন অফ বাংলাদেশ,Sports,Male,0.0,not bully
4,পটকা মাছ,Politician,Male,0.0,troll


### `multilabel_12557/Multilablel_Cyberbully_Data.csv`

,Gender,Profession,comment,bully,sexual,religious,threat,spam
0,female,dancer,এই দেশে এইসব কি হচ্ছে,0,0,0,0,0
1,female,dancer,মানে কি বলব,0,0,0,0,0
2,female,dancer,ভাই ভিডিও ফুল প্লিজ,0,1,0,0,1
3,female,dancer,নিজের খরচ নিজেই চালাতে পারবেন এমন ভালো একটা জব আছে ঘরে বসে করতে পারবেন,0,0,0,0,1
4,female,dancer,ভিডিও কলে রেগুলার কাজ করতে পারবেন,1,1,0,0,1


## 6. High-level summary by dataset folder


In [8]:
summary_df = (
    inventory_df.groupby('dataset_folder', dropna=False)
    .agg(
        file_count=('filename', 'count'),
        total_rows=('rows', 'sum'),
        total_missing=('missing_values_total', 'sum'),
        total_duplicates=('duplicate_rows', 'sum')
    )
    .reset_index()
)

display(summary_df)


,dataset_folder,file_count,total_rows,total_missing,total_duplicates
0,banth,4,73999,1696,0
1,bd_shs,1,5029,5226,0
2,facebook_44001,1,44001,3,140
3,multilabel_12557,1,12546,0,1291


## 7. Save outputs


In [9]:
inventory_csv = OUTPUT_DIR / 'dataset_inventory_summary.csv'
files_csv = OUTPUT_DIR / 'dataset_file_tree.csv'
summary_csv = OUTPUT_DIR / 'dataset_folder_summary.csv'

inventory_df.to_csv(inventory_csv, index=False, encoding='utf-8-sig')
files_df.to_csv(files_csv, index=False, encoding='utf-8-sig')
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')

print('Saved:', inventory_csv)
print('Saved:', files_csv)
print('Saved:', summary_csv)


Saved: /kaggle/working/CyberBully_Detection_Paper/outputs/dataset_inventory_summary.csv
Saved: /kaggle/working/CyberBully_Detection_Paper/outputs/dataset_file_tree.csv
Saved: /kaggle/working/CyberBully_Detection_Paper/outputs/dataset_folder_summary.csv
